In [ ]:
#TAME/
#├── selected_participants_20.csv
#├── selected_audio_files_20_selection.csv
#├── selected_audio_files_20_original.csv
#├── selected_audio_files_20_wiener.csv
#├── selected_audio_files_20_pain_balanced_selection.csv
#├── selected_audio_files_20_pain_balanced_original.csv
#├── selected_audio_files_20_pain_balanced_wiener.csv

# audio files (.wav)
#├── selected_original_audio_20/
#├── selected_wiener_audio_20/
#├── selected_original_audio_20_pain_balanced/
#└── selected_wiener_audio_20_pain_balanced/

# perturbations (original)
#├── original_intensity_perturbations_pain/
#│   ├── -6dB/
#│   ├── -3dB/
#│   ├── +3dB/
#│   └── +6dB/
#├── original_gaussian_noise_perturbations_pain/
#│   ├── low_gaussian_noise/
#│   ├── medium_gaussian_noise/
#│   ├── high_gaussian_noise/
#│   └── very_high_gaussian_noise/
#├── original_pink_noise_perturbations_pain/
#│   ├── low_pink_noise/
#│   ├── medium_pink_noise/
#│   ├── high_pink_noise/
#│   └── very_high_pink_noise/
#└── original_lowpass_perturbations_pain/
#    ├── low_lowpass/
#    ├── medium_lowpass/
#    ├── high_lowpass/
#    └── very_high_lowpass/

# perturbations (wiener)
#├── wiener_intensity_perturbations_pain/
#│   ├── -6dB/
#│   ├── -3dB/
#│   ├── +3dB/
#│   └── +6dB/
#├── wiener_gaussian_noise_perturbations_pain/
#│   ├── low_gaussian_noise/
#│   ├── medium_gaussian_noise/
#│   ├── high_gaussian_noise/
#│   └── very_high_gaussian_noise/
#├── wiener_pink_noise_perturbations_pain/
#│   ├── low_pink_noise/
#│   ├── medium_pink_noise/
#│   ├── high_pink_noise/
#│   └── very_high_pink_noise/
#└── wiener_lowpass_perturbations_pain/
#    ├── low_lowpass/
#    ├── medium_lowpass/
#    ├── high_lowpass/
#    └── very_high_lowpass/

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
from scipy.io import wavfile
from scipy.signal import butter, sosfiltfilt

#input wiener pain balanced 20 participants:
#selected_wiener_audio_20_pain_balanced

#input original pain balanced 20 participants:
#selected_original_audio_20_pain_balanced

#output original:
#original_gaussian_noise_perturbations_pain
#original_lowpass_perturbations_pain
#original_pink_noise_perturbations_pain
#original_intensity_perturbations_pain

#output wiener:
#wiener_intensity_perturbations_pain
#wiener_gaussian_noise_perturbations_pain
#wiener_pink_noise_perturbations_pain
#wiener_lowpass_perturbations_pain

# Paths
DATA_PATH = Path(r"C:\Users\marti\Documents\Technical Medicine Master\Stages TM2\TM2-3\Technische opdracht\TAME")
#WIENER_INPUT_DIR = DATA_PATH / "selected_wiener_audio_20_pain_balanced"
ORIGINAL_INPUT_DIR = DATA_PATH / "selected_original_audio_20_pain_balanced"

# Perturbation config
PERTURBATIONS = {
    "intensity": {
        "output_dir": DATA_PATH / "original_intensity_perturbations_pain",
        "value_key": "gain_db",
        "levels": {
            "-6dB": -6,
            "-3dB": -3,
            "+3dB": 3,
            "+6dB": 6,
        },
    },
    "gaussian_noise": {
        "output_dir": DATA_PATH / "original_gaussian_noise_perturbations_pain",
        "value_key": "noise_std_fraction",
        "levels": {
            "low_gaussian_noise": 0.005,
            "medium_gaussian_noise": 0.02,
            "high_gaussian_noise": 0.05,
            "very_high_gaussian_noise": 0.1,
        },
    },
    "pink_noise": {
        "output_dir": DATA_PATH / "original_pink_noise_perturbations_pain",
        "value_key": "noise_std_fraction",
        "levels": {
            "low_pink_noise": 0.005,
            "medium_pink_noise": 0.02,
            "high_pink_noise": 0.05,
            "very_high_pink_noise": 0.1,
        },
    },
    "lowpass": {
        "output_dir": DATA_PATH / "original_lowpass_perturbations_pain",
        "value_key": "cutoff_hz",
        "levels": {
            "low_lowpass": 6000,
            "medium_lowpass": 5000,
            "high_lowpass": 4000,
            "very_high_lowpass": 3000,
        },
        "filter_order": 4,
    },
}


def collect_wav_files(root_dir):
    return [
        os.path.join(root, file)
        for root, _, files in os.walk(root_dir)
        for file in files
        if file.endswith(".wav")
    ]


def load_wav_file(file_path):
    return wavfile.read(file_path)


def save_wav_file(file_path, sample_rate, signal):
    signal = np.clip(signal, -32768, 32767).astype(np.int16)
    wavfile.write(file_path, sample_rate, signal)


def make_output_path(input_path, input_root, output_root, perturbation_name):
    relative_path = os.path.relpath(input_path, input_root)
    output_path = os.path.join(output_root, perturbation_name, relative_path)
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    return output_path


def generate_pink_noise(n):
    white = np.random.randn(n)
    pink_fft = np.fft.rfft(white)
    freqs = np.fft.rfftfreq(n)

    if len(freqs) > 1:
        freqs[0] = freqs[1]

    pink_fft = pink_fft / np.sqrt(freqs)
    pink = np.fft.irfft(pink_fft, n=n)
    return pink


def apply_lowpass_filter(signal, sample_rate, cutoff_hz, filter_order=4):
    signal = signal.astype(np.float32)
    nyquist = sample_rate / 2

    if cutoff_hz >= nyquist:
        raise ValueError(
            f"cutoff_hz ({cutoff_hz}) must be smaller than Nyquist frequency ({nyquist} Hz)"
        )

    sos = butter(
        N=filter_order,
        Wn=cutoff_hz,
        btype="low",
        fs=sample_rate,
        output="sos"
    )

    return sosfiltfilt(sos, signal)


def apply_perturbation(signal, sample_rate, perturbation_type, value, config):
    signal = signal.astype(np.float32)

    if perturbation_type == "intensity":
        factor = 10 ** (value / 20.0)
        return signal * factor

    if perturbation_type == "gaussian_noise":
        noise_std = value * np.std(signal)
        noise = np.random.normal(0.0, noise_std, size=signal.shape)
        return signal + noise

    if perturbation_type == "pink_noise":
        signal_std = np.std(signal)
        pink_noise = generate_pink_noise(len(signal))
        pink_noise = pink_noise / np.std(pink_noise)
        noise_std = value * signal_std
        pink_noise = pink_noise * noise_std
        return signal + pink_noise

    if perturbation_type == "lowpass":
        filter_order = config.get("filter_order", 4)
        return apply_lowpass_filter(
            signal=signal,
            sample_rate=sample_rate,
            cutoff_hz=value,
            filter_order=filter_order
        )

    raise ValueError(f"Unknown perturbation type: {perturbation_type}")


def process_all_perturbations(audio_files, input_root, perturbation_config):
    all_results = {}

    for perturbation_type, config in perturbation_config.items():
        output_dir = config["output_dir"]
        value_key = config["value_key"]
        levels = config["levels"]

        output_dir.mkdir(parents=True, exist_ok=True)
        processed_rows = []

        for input_path in audio_files:
            try:
                sample_rate, signal = load_wav_file(input_path)
                participant_id = os.path.basename(os.path.dirname(input_path))
                filename = os.path.basename(input_path)

                for perturbation_name, value in levels.items():
                    perturbed_signal = apply_perturbation(
                        signal=signal,
                        sample_rate=sample_rate,
                        perturbation_type=perturbation_type,
                        value=value,
                        config=config
                    )

                    output_path = make_output_path(
                        input_path=input_path,
                        input_root=input_root,
                        output_root=output_dir,
                        perturbation_name=perturbation_name
                    )

                    save_wav_file(output_path, sample_rate, perturbed_signal)

                    row = {
                        "participant_id": participant_id,
                        "filename": filename,
                        "original_wiener_file_path": input_path,
                        "perturbation_type": perturbation_type,
                        "perturbation": perturbation_name,
                        value_key: value,
                        "perturbed_file_path": output_path,
                    }

                    if perturbation_type == "lowpass":
                        row["filter_order"] = config.get("filter_order", 4)

                    processed_rows.append(row)

            except Exception as e:
                print(f"Error processing {input_path}: {e}")

        all_results[perturbation_type] = pd.DataFrame(processed_rows)

    return all_results


# Run
#wiener_audio_files = collect_wav_files(WIENER_INPUT_DIR)
original_audio_files = collect_wav_files(ORIGINAL_INPUT_DIR)
print(f"Number of Original audio files: {len(original_audio_files)}")

results = process_all_perturbations(
    audio_files=original_audio_files,
    #input_root=WIENER_INPUT_DIR,
    input_root=ORIGINAL_INPUT_DIR,
    perturbation_config=PERTURBATIONS,
)

df_intensity = results["intensity"]
df_gaussian = results["gaussian_noise"]
df_pink = results["pink_noise"]
df_lowpass = results["lowpass"]

print("Finished processing all perturbations.")

Number of Original audio files: 1055
Finished processing all perturbations.


In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
from scipy.io import wavfile
from scipy.signal import butter, sosfiltfilt

# =========================
# Paths
# =========================
DATA_PATH = Path(r"C:\Users\marti\Documents\Technical Medicine Master\Stages TM2\TM2-3\Technische opdracht\TAME")

INPUT_DATASETS = {

    # Pain balanced 
  
    "original": {
        "input_dir": DATA_PATH / "selected_original_audio_20_pain_balanced",
        "output_prefix": "original",
    },
    "wiener": {
        "input_dir": DATA_PATH / "selected_wiener_audio_20_pain_balanced",
        "output_prefix": "wiener",
    },
}

# =========================
# Perturbation config
# =========================
PERTURBATION_TEMPLATES = {
    "intensity": {
        "value_key": "gain_db",
        "levels": {
            "-6dB": -6,
            "-3dB": -3,
            "+3dB": 3,
            "+6dB": 6,
        },
    },
    "gaussian_noise": {
        "value_key": "noise_std_fraction",
        "levels": {
            "low_gaussian_noise": 0.005,
            "medium_gaussian_noise": 0.02,
            "high_gaussian_noise": 0.05,
            "very_high_gaussian_noise": 0.1,
        },
    },
    "pink_noise": {
        "value_key": "noise_std_fraction",
        "levels": {
            "low_pink_noise": 0.005,
            "medium_pink_noise": 0.02,
            "high_pink_noise": 0.05,
            "very_high_pink_noise": 0.1,
        },
    },
    "lowpass": {
        "value_key": "cutoff_hz",
        "levels": {
            "low_lowpass": 6000,
            "medium_lowpass": 5000,
            "high_lowpass": 4000,
            "very_high_lowpass": 3000,
        },
        "filter_order": 4,
    },
}


# =========================
# Helpers
# =========================
def collect_wav_files(root_dir):
    return [
        os.path.join(root, file)
        for root, _, files in os.walk(root_dir)
        for file in files
        if file.endswith(".wav")
    ]


def load_wav_file(file_path):
    return wavfile.read(file_path)


def save_wav_file(file_path, sample_rate, signal):
    signal = np.clip(signal, -32768, 32767).astype(np.int16)
    wavfile.write(file_path, sample_rate, signal)


def make_output_path(input_path, input_root, output_root, perturbation_name):
    relative_path = os.path.relpath(input_path, input_root)
    output_path = os.path.join(output_root, perturbation_name, relative_path)
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    return output_path


def generate_pink_noise(n):
    white = np.random.randn(n)
    pink_fft = np.fft.rfft(white)
    freqs = np.fft.rfftfreq(n)

    if len(freqs) > 1:
        freqs[0] = freqs[1]

    pink_fft = pink_fft / np.sqrt(freqs)
    pink = np.fft.irfft(pink_fft, n=n)
    return pink


def apply_lowpass_filter(signal, sample_rate, cutoff_hz, filter_order=4):
    signal = signal.astype(np.float32)
    nyquist = sample_rate / 2

    if cutoff_hz >= nyquist:
        raise ValueError(
            f"cutoff_hz ({cutoff_hz}) must be smaller than Nyquist frequency ({nyquist} Hz)"
        )

    sos = butter(
        N=filter_order,
        Wn=cutoff_hz,
        btype="low",
        fs=sample_rate,
        output="sos"
    )

    return sosfiltfilt(sos, signal)


def apply_perturbation(signal, sample_rate, perturbation_type, value, config):
    signal = signal.astype(np.float32)

    if perturbation_type == "intensity":
        factor = 10 ** (value / 20.0)
        return signal * factor

    if perturbation_type == "gaussian_noise":
        noise_std = value * np.std(signal)
        noise = np.random.normal(0.0, noise_std, size=signal.shape)
        return signal + noise

    if perturbation_type == "pink_noise":
        signal_std = np.std(signal)
        pink_noise = generate_pink_noise(len(signal))
        pink_std = np.std(pink_noise)

        if pink_std > 0:
            pink_noise = pink_noise / pink_std
        else:
            pink_noise = np.zeros_like(signal, dtype=np.float32)

        noise_std = value * signal_std
        pink_noise = pink_noise * noise_std
        return signal + pink_noise

    if perturbation_type == "lowpass":
        filter_order = config.get("filter_order", 4)
        return apply_lowpass_filter(
            signal=signal,
            sample_rate=sample_rate,
            cutoff_hz=value,
            filter_order=filter_order
        )

    raise ValueError(f"Unknown perturbation type: {perturbation_type}")


def get_baseline_dataframe(audio_files, dataset_name):
    rows = []

    for input_path in audio_files:
        participant_id = os.path.basename(os.path.dirname(input_path))
        filename = os.path.basename(input_path)

        rows.append({
            "dataset": dataset_name,
            "participant_id": participant_id,
            "filename": filename,
            "baseline_file_path": input_path,
            "perturbation_type": "none",
            "perturbation": "none",
        })

    return pd.DataFrame(rows)


def process_all_perturbations(audio_files, input_root, perturbation_config, dataset_name):
    all_results = {}

    for perturbation_type, config in perturbation_config.items():
        output_dir = config["output_dir"]
        value_key = config["value_key"]
        levels = config["levels"]

        output_dir.mkdir(parents=True, exist_ok=True)
        processed_rows = []

        for input_path in audio_files:
            try:
                sample_rate, signal = load_wav_file(input_path)
                participant_id = os.path.basename(os.path.dirname(input_path))
                filename = os.path.basename(input_path)

                for perturbation_name, value in levels.items():
                    perturbed_signal = apply_perturbation(
                        signal=signal,
                        sample_rate=sample_rate,
                        perturbation_type=perturbation_type,
                        value=value,
                        config=config
                    )

                    output_path = make_output_path(
                        input_path=input_path,
                        input_root=input_root,
                        output_root=output_dir,
                        perturbation_name=perturbation_name
                    )

                    save_wav_file(output_path, sample_rate, perturbed_signal)

                    row = {
                        "dataset": dataset_name,
                        "participant_id": participant_id,
                        "filename": filename,
                        "baseline_file_path": input_path,
                        "perturbation_type": perturbation_type,
                        "perturbation": perturbation_name,
                        value_key: value,
                        "perturbed_file_path": output_path,
                    }

                    if perturbation_type == "lowpass":
                        row["filter_order"] = config.get("filter_order", 4)

                    processed_rows.append(row)

            except Exception as e:
                print(f"Error processing {input_path}: {e}")

        all_results[perturbation_type] = pd.DataFrame(processed_rows)

    return all_results


def build_dataset_perturbation_config(dataset_prefix):
    config = {}

    for perturbation_type, template in PERTURBATION_TEMPLATES.items():
        config[perturbation_type] = template.copy()
        config[perturbation_type]["output_dir"] = DATA_PATH / f"{dataset_prefix}_{perturbation_type}_perturbations_pain"

    return config


# =========================
# Run for original + wiener
# =========================
all_baselines = {}
all_results = {}

for dataset_name, dataset_info in INPUT_DATASETS.items():
    input_dir = dataset_info["input_dir"]
    output_prefix = dataset_info["output_prefix"]

    audio_files = collect_wav_files(input_dir)
    print(f"Number of {dataset_name} audio files: {len(audio_files)}")

    # 1. Baseline dataframe (zonder perturbation)
    baseline_df = get_baseline_dataframe(audio_files, dataset_name)
    all_baselines[dataset_name] = baseline_df

    # optioneel opslaan
    baseline_df.to_csv(DATA_PATH / f"opensmile_input_{dataset_name}_baseline.csv", index=False)

    # 2. Perturbation config voor deze dataset
    perturbation_config = build_dataset_perturbation_config(output_prefix)

    # 3. Perturbations maken
    results = process_all_perturbations(
        audio_files=audio_files,
        input_root=input_dir,
        perturbation_config=perturbation_config,
        dataset_name=dataset_name,
    )

    all_results[dataset_name] = results

    # optioneel opslaan per perturbatie
    for perturbation_type, df in results.items():
        df.to_csv(DATA_PATH / f"{dataset_name}_{perturbation_type}_perturbations_overview.csv", index=False)

print("Finished processing baseline + all perturbations.")

# Handige losse dataframes
df_original_baseline = all_baselines["original"]
df_wiener_baseline = all_baselines["wiener"]

df_original_intensity = all_results["original"]["intensity"]
df_original_gaussian = all_results["original"]["gaussian_noise"]
df_original_pink = all_results["original"]["pink_noise"]
df_original_lowpass = all_results["original"]["lowpass"]

df_wiener_intensity = all_results["wiener"]["intensity"]
df_wiener_gaussian = all_results["wiener"]["gaussian_noise"]
df_wiener_pink = all_results["wiener"]["pink_noise"]
df_wiener_lowpass = all_results["wiener"]["lowpass"]

Number of original audio files: 1055
Number of wiener audio files: 1055
Finished processing baseline + all perturbations.


Code with groups 

In [1]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
from scipy.io import wavfile
from scipy.signal import butter, sosfiltfilt

# =========================
# Paths
# =========================
DATA_PATH = Path(r"C:\Users\marti\Documents\Technical Medicine Master\Stages TM2\TM2-3\Technische opdracht\TAME")

INPUT_DATASETS = {
    # Pain balanced
    "original": {
        "input_dir": DATA_PATH / "selected_original_audio_20_pain_balanced",
        "output_prefix": "original",
    },
    "wiener": {
        "input_dir": DATA_PATH / "selected_wiener_audio_20_pain_balanced",
        "output_prefix": "wiener",
    },

    # Pain groups
    "original_pain_group_1_to_4": {
        "input_dir": DATA_PATH / "selected_original_audio_20_pain_group_1_to_4",
        "output_prefix": "original_pain_group_1_to_4",
    },
    "wiener_pain_group_1_to_4": {
        "input_dir": DATA_PATH / "selected_wiener_audio_20_pain_group_1_to_4",
        "output_prefix": "wiener_pain_group_1_to_4",
    },

    "original_pain_group_5_to_6": {
        "input_dir": DATA_PATH / "selected_original_audio_20_pain_group_5_to_6",
        "output_prefix": "original_pain_group_5_to_6",
    },
    "wiener_pain_group_5_to_6": {
        "input_dir": DATA_PATH / "selected_wiener_audio_20_pain_group_5_to_6",
        "output_prefix": "wiener_pain_group_5_to_6",
    },

    "original_pain_group_7_to_10": {
        "input_dir": DATA_PATH / "selected_original_audio_20_pain_group_7_to_10",
        "output_prefix": "original_pain_group_7_to_10",
    },
    "wiener_pain_group_7_to_10": {
        "input_dir": DATA_PATH / "selected_wiener_audio_20_pain_group_7_to_10",
        "output_prefix": "wiener_pain_group_7_to_10",
    },
}

# =========================
# Perturbation config
# =========================
PERTURBATION_TEMPLATES = {
    "intensity": {
        "value_key": "gain_db",
        "levels": {
            "-6dB": -6,
            "-3dB": -3,
            "+3dB": 3,
            "+6dB": 6,
        },
    },
    "gaussian_noise": {
        "value_key": "noise_std_fraction",
        "levels": {
            "low_gaussian_noise": 0.005,
            "medium_gaussian_noise": 0.02,
            "high_gaussian_noise": 0.05,
            "very_high_gaussian_noise": 0.1,
        },
    },
    "pink_noise": {
        "value_key": "noise_std_fraction",
        "levels": {
            "low_pink_noise": 0.005,
            "medium_pink_noise": 0.02,
            "high_pink_noise": 0.05,
            "very_high_pink_noise": 0.1,
        },
    },
    "lowpass": {
        "value_key": "cutoff_hz",
        "levels": {
            "low_lowpass": 6000,
            "medium_lowpass": 5000,
            "high_lowpass": 4000,
            "very_high_lowpass": 3000,
        },
        "filter_order": 4,
    },
}

# =========================
# Helpers
# =========================
def collect_wav_files(root_dir):
    return [
        os.path.join(root, file)
        for root, _, files in os.walk(root_dir)
        for file in files
        if file.endswith(".wav")
    ]


def load_wav_file(file_path):
    return wavfile.read(file_path)


def save_wav_file(file_path, sample_rate, signal):
    signal = np.clip(signal, -32768, 32767).astype(np.int16)
    wavfile.write(file_path, sample_rate, signal)


def make_output_path(input_path, input_root, output_root, perturbation_name):
    relative_path = os.path.relpath(input_path, input_root)
    output_path = os.path.join(output_root, perturbation_name, relative_path)
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    return output_path


def generate_pink_noise(n):
    white = np.random.randn(n)
    pink_fft = np.fft.rfft(white)
    freqs = np.fft.rfftfreq(n)

    if len(freqs) > 1:
        freqs[0] = freqs[1]

    pink_fft = pink_fft / np.sqrt(freqs)
    pink = np.fft.irfft(pink_fft, n=n)
    return pink


def apply_lowpass_filter(signal, sample_rate, cutoff_hz, filter_order=4):
    signal = signal.astype(np.float32)
    nyquist = sample_rate / 2

    if cutoff_hz >= nyquist:
        raise ValueError(
            f"cutoff_hz ({cutoff_hz}) must be smaller than Nyquist frequency ({nyquist} Hz)"
        )

    sos = butter(
        N=filter_order,
        Wn=cutoff_hz,
        btype="low",
        fs=sample_rate,
        output="sos"
    )

    return sosfiltfilt(sos, signal)


def apply_perturbation(signal, sample_rate, perturbation_type, value, config):
    signal = signal.astype(np.float32)

    if perturbation_type == "intensity":
        factor = 10 ** (value / 20.0)
        return signal * factor

    if perturbation_type == "gaussian_noise":
        noise_std = value * np.std(signal)
        noise = np.random.normal(0.0, noise_std, size=signal.shape)
        return signal + noise

    if perturbation_type == "pink_noise":
        signal_std = np.std(signal)
        pink_noise = generate_pink_noise(len(signal))
        pink_std = np.std(pink_noise)

        if pink_std > 0:
            pink_noise = pink_noise / pink_std
        else:
            pink_noise = np.zeros_like(signal, dtype=np.float32)

        noise_std = value * signal_std
        pink_noise = pink_noise * noise_std
        return signal + pink_noise

    if perturbation_type == "lowpass":
        filter_order = config.get("filter_order", 4)
        return apply_lowpass_filter(
            signal=signal,
            sample_rate=sample_rate,
            cutoff_hz=value,
            filter_order=filter_order
        )

    raise ValueError(f"Unknown perturbation type: {perturbation_type}")


def get_baseline_dataframe(audio_files, dataset_name):
    rows = []

    for input_path in audio_files:
        participant_id = os.path.basename(os.path.dirname(input_path))
        filename = os.path.basename(input_path)

        rows.append({
            "dataset": dataset_name,
            "participant_id": participant_id,
            "filename": filename,
            "baseline_file_path": input_path,
            "perturbation_type": "none",
            "perturbation": "none",
        })

    return pd.DataFrame(rows)


def process_all_perturbations(audio_files, input_root, perturbation_config, dataset_name):
    all_results = {}

    for perturbation_type, config in perturbation_config.items():
        output_dir = config["output_dir"]
        value_key = config["value_key"]
        levels = config["levels"]

        output_dir.mkdir(parents=True, exist_ok=True)
        processed_rows = []

        for input_path in audio_files:
            try:
                sample_rate, signal = load_wav_file(input_path)
                participant_id = os.path.basename(os.path.dirname(input_path))
                filename = os.path.basename(input_path)

                for perturbation_name, value in levels.items():
                    perturbed_signal = apply_perturbation(
                        signal=signal,
                        sample_rate=sample_rate,
                        perturbation_type=perturbation_type,
                        value=value,
                        config=config
                    )

                    output_path = make_output_path(
                        input_path=input_path,
                        input_root=input_root,
                        output_root=output_dir,
                        perturbation_name=perturbation_name
                    )

                    save_wav_file(output_path, sample_rate, perturbed_signal)

                    row = {
                        "dataset": dataset_name,
                        "participant_id": participant_id,
                        "filename": filename,
                        "baseline_file_path": input_path,
                        "perturbation_type": perturbation_type,
                        "perturbation": perturbation_name,
                        value_key: value,
                        "perturbed_file_path": output_path,
                    }

                    if perturbation_type == "lowpass":
                        row["filter_order"] = config.get("filter_order", 4)

                    processed_rows.append(row)

            except Exception as e:
                print(f"Error processing {input_path}: {e}")

        all_results[perturbation_type] = pd.DataFrame(processed_rows)

    return all_results


def build_dataset_perturbation_config(dataset_prefix):
    config = {}

    for perturbation_type, template in PERTURBATION_TEMPLATES.items():
        config[perturbation_type] = template.copy()
        config[perturbation_type]["output_dir"] = DATA_PATH / f"{dataset_prefix}_{perturbation_type}_perturbations_pain"

    return config


# =========================
# Run for all datasets
# =========================
all_baselines = {}
all_results = {}

for dataset_name, dataset_info in INPUT_DATASETS.items():
    input_dir = dataset_info["input_dir"]
    output_prefix = dataset_info["output_prefix"]

    audio_files = collect_wav_files(input_dir)
    print(f"Number of {dataset_name} audio files: {len(audio_files)}")

    baseline_df = get_baseline_dataframe(audio_files, dataset_name)
    all_baselines[dataset_name] = baseline_df
    baseline_df.to_csv(DATA_PATH / f"opensmile_input_{dataset_name}_baseline.csv", index=False)

    perturbation_config = build_dataset_perturbation_config(output_prefix)

    results = process_all_perturbations(
        audio_files=audio_files,
        input_root=input_dir,
        perturbation_config=perturbation_config,
        dataset_name=dataset_name,
    )

    all_results[dataset_name] = results

    for perturbation_type, df in results.items():
        df.to_csv(DATA_PATH / f"{dataset_name}_{perturbation_type}_perturbations_overview.csv", index=False)

print("Finished processing baseline + all perturbations.")

# =========================
# Handy separate dataframes
# =========================
df_original_baseline = all_baselines["original"]
df_wiener_baseline = all_baselines["wiener"]

df_original_intensity = all_results["original"]["intensity"]
df_original_gaussian = all_results["original"]["gaussian_noise"]
df_original_pink = all_results["original"]["pink_noise"]
df_original_lowpass = all_results["original"]["lowpass"]

df_wiener_intensity = all_results["wiener"]["intensity"]
df_wiener_gaussian = all_results["wiener"]["gaussian_noise"]
df_wiener_pink = all_results["wiener"]["pink_noise"]
df_wiener_lowpass = all_results["wiener"]["lowpass"]

# pain group 1-4
df_original_pain_group_1_to_4_baseline = all_baselines["original_pain_group_1_to_4"]
df_wiener_pain_group_1_to_4_baseline = all_baselines["wiener_pain_group_1_to_4"]

df_original_pain_group_1_to_4_intensity = all_results["original_pain_group_1_to_4"]["intensity"]
df_original_pain_group_1_to_4_gaussian = all_results["original_pain_group_1_to_4"]["gaussian_noise"]
df_original_pain_group_1_to_4_pink = all_results["original_pain_group_1_to_4"]["pink_noise"]
df_original_pain_group_1_to_4_lowpass = all_results["original_pain_group_1_to_4"]["lowpass"]

df_wiener_pain_group_1_to_4_intensity = all_results["wiener_pain_group_1_to_4"]["intensity"]
df_wiener_pain_group_1_to_4_gaussian = all_results["wiener_pain_group_1_to_4"]["gaussian_noise"]
df_wiener_pain_group_1_to_4_pink = all_results["wiener_pain_group_1_to_4"]["pink_noise"]
df_wiener_pain_group_1_to_4_lowpass = all_results["wiener_pain_group_1_to_4"]["lowpass"]

# pain group 5-6
df_original_pain_group_5_to_6_baseline = all_baselines["original_pain_group_5_to_6"]
df_wiener_pain_group_5_to_6_baseline = all_baselines["wiener_pain_group_5_to_6"]

df_original_pain_group_5_to_6_intensity = all_results["original_pain_group_5_to_6"]["intensity"]
df_original_pain_group_5_to_6_gaussian = all_results["original_pain_group_5_to_6"]["gaussian_noise"]
df_original_pain_group_5_to_6_pink = all_results["original_pain_group_5_to_6"]["pink_noise"]
df_original_pain_group_5_to_6_lowpass = all_results["original_pain_group_5_to_6"]["lowpass"]

df_wiener_pain_group_5_to_6_intensity = all_results["wiener_pain_group_5_to_6"]["intensity"]
df_wiener_pain_group_5_to_6_gaussian = all_results["wiener_pain_group_5_to_6"]["gaussian_noise"]
df_wiener_pain_group_5_to_6_pink = all_results["wiener_pain_group_5_to_6"]["pink_noise"]
df_wiener_pain_group_5_to_6_lowpass = all_results["wiener_pain_group_5_to_6"]["lowpass"]

# pain group 7-10
df_original_pain_group_7_to_10_baseline = all_baselines["original_pain_group_7_to_10"]
df_wiener_pain_group_7_to_10_baseline = all_baselines["wiener_pain_group_7_to_10"]

df_original_pain_group_7_to_10_intensity = all_results["original_pain_group_7_to_10"]["intensity"]
df_original_pain_group_7_to_10_gaussian = all_results["original_pain_group_7_to_10"]["gaussian_noise"]
df_original_pain_group_7_to_10_pink = all_results["original_pain_group_7_to_10"]["pink_noise"]
df_original_pain_group_7_to_10_lowpass = all_results["original_pain_group_7_to_10"]["lowpass"]

df_wiener_pain_group_7_to_10_intensity = all_results["wiener_pain_group_7_to_10"]["intensity"]
df_wiener_pain_group_7_to_10_gaussian = all_results["wiener_pain_group_7_to_10"]["gaussian_noise"]
df_wiener_pain_group_7_to_10_pink = all_results["wiener_pain_group_7_to_10"]["pink_noise"]
df_wiener_pain_group_7_to_10_lowpass = all_results["wiener_pain_group_7_to_10"]["lowpass"]

Number of original audio files: 1055
Number of wiener audio files: 1055
Number of original_pain_group_1_to_4 audio files: 200
Number of wiener_pain_group_1_to_4 audio files: 200
Number of original_pain_group_5_to_6 audio files: 200
Number of wiener_pain_group_5_to_6 audio files: 200
Number of original_pain_group_7_to_10 audio files: 200
Number of wiener_pain_group_7_to_10 audio files: 200
Finished processing baseline + all perturbations.
